# Holdout evaluation (one time)

Each model is fit **once** on all dev rows of `splits.csv` (same features, hyperparameters and fixed CNN epochs as `model_comparison.py`), then scored on the internal test split (23 signer groups never used in CV) and, optionally, the external 686-signature set.

Nothing here feeds back into model choice: the selected model is fixed in advance in `SELECTED`.

Needs `model_comparison.py` in this folder (it holds the notebook's feature and model code). The CNN runs only if PyTorch is installed.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

sys.path.insert(0, str(Path.cwd()))  # model_comparison.py lives next to this notebook
import model_comparison as mc  # noqa: E402

In [2]:
CNN_SEEDS = (42, 43, 44)


def metrics(y, s, thr):
    return {"n": len(y), "prevalence": y.mean(),
            "pr_auc": average_precision_score(y, s),
            "roc_auc": roc_auc_score(y, s),
            "f1": f1_score(y, (s > thr).astype(int), zero_division=0)}


def cluster_boot(y, s, groups, n_boot, rng):
    """95% CI for PR-AUC and ROC-AUC, resampling whole signer groups."""
    ug = np.unique(groups)
    idx_by_g = [np.where(groups == g)[0] for g in ug]
    pr, roc = [], []
    for _ in range(n_boot):
        idx = np.concatenate([idx_by_g[i] for i in rng.integers(0, len(ug), len(ug))])
        if len(np.unique(y[idx])) < 2:
            continue
        pr.append(average_precision_score(y[idx], s[idx]))
        roc.append(roc_auc_score(y[idx], s[idx]))
    q = lambda a: np.percentile(a, [2.5, 97.5])
    return q(pr), q(roc)


def load_external(xlsx, img_dir, dev_names):
    r = pd.read_excel(xlsx, sheet_name="Relabeled")
    r = r[r["FINAL LABEL"].isin(["Legible", "Illegible"])].reset_index(drop=True)
    y = (r["FINAL LABEL"] == "Legible").astype(int).to_numpy()
    imgs = np.zeros((len(r), 256, 256), dtype=np.uint8)
    for i, f in enumerate(r["File Name"]):
        im = Image.open(Path(img_dir) / f).convert("L")
        imgs[i] = np.asarray(im.resize((256, 256), Image.BOX))   # same as load_images()
    norm = lambda n: re.sub("[^a-z]", "", str(n).lower()) if pd.notna(n) else ""
    cols = ["Name for TrOCR check", "Anthony name", "Claude name (blind read)"]
    shared = r[cols].apply(lambda row: any(norm(v) in dev_names for v in row if norm(v)), axis=1)
    # no signer ids exist here, so each image is its own bootstrap unit
    return r, y, imgs, shared.to_numpy()

In [3]:
def run_holdout(args):

    wd = Path(args.workdir)
    splits = pd.read_csv(wd / "splits.csv")
    imgs = mc.load_images(wd, splits)
    dev_m = (splits["split"] == "dev").to_numpy()
    te_m = (splits["split"] == "test").to_numpy()
    dev, te = splits[dev_m].reset_index(drop=True), splits[te_m].reset_index(drop=True)
    assert set(dev["group"]).isdisjoint(set(te["group"])), "signer groups overlap"
    y_dev, y_te = dev["label"].to_numpy(), te["label"].to_numpy()
    print(f"Fit on dev: {len(dev)} rows / {dev.group.nunique()} groups; "
          f"test: {len(te)} rows / {te.group.nunique()} groups, legible {y_te.mean():.3f}")

    ext = None
    if args.external_xlsx:
        r, y_ext, imgs_ext, shared = load_external(args.external_xlsx, args.external_dir,
                                                   set(dev["name"]))
        ext = dict(r=r, y=y_ext, imgs=imgs_ext, shared=shared)
        print(f"External: {len(r)} usable rows, legible {y_ext.mean():.3f}; "
              f"{shared.sum()} share a name with a dev signer")

    test_scores, ext_scores, thr_of = {}, {}, {}

    # ---- classical ----------------------------------------------------------
    sets = {"dev": imgs[dev_m], "test": imgs[te_m]}
    if ext:
        sets["ext"] = ext["imgs"]
    feats = {k: mc.classical_features(mc.make_variant(v, "norm", args.img_size)) for k, v in sets.items()}
    contr = {k: mc.contrast_feature(v) for k, v in sets.items()}
    for name, make in mc.classical_models().items():
        F = contr if name == "Logistic (contrast only)" else feats
        m = make().fit(F["dev"], y_dev)
        test_scores[name], thr_of[name] = mc.score_and_threshold(m, F["test"])
        if ext:
            ext_scores[name] = mc.score_and_threshold(m, F["ext"])[0]
        print(f"  {name} fit and scored")

    # ---- CNN: one model per seed, metrics averaged over seeds ---------------
    if not args.skip_cnn:
        try:
            import torch
        except ImportError:
            torch = None
            print("PyTorch not installed; skipping the CNN.")
        if torch is not None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            for v in [x.strip() for x in args.cnn_variants.split(",")]:
                Xd = mc.make_variant(imgs[dev_m], v, args.img_size)
                Xt = mc.make_variant(imgs[te_m], v, args.img_size)
                Xe = mc.make_variant(ext["imgs"], v, args.img_size) if ext else None
                for sd in CNN_SEEDS:
                    Xeval = np.concatenate([Xt, Xe]) if ext else Xt
                    p = mc.run_cnn_fold(Xd, y_dev, Xeval, args, device, sd)
                    name = f"CNN ({v})__s{sd}"
                    test_scores[name], thr_of[name] = p[:len(Xt)], 0.5
                    if ext:
                        ext_scores[name] = p[len(Xt):]
                    print(f"  {name} done")

    # ---- metrics ------------------------------------------------------------
    rng = np.random.default_rng(0)
    groups = te["group"].to_numpy()
    rows, fam_rows = [], []
    for name, s in test_scores.items():
        r = {"model": name, **metrics(y_te, s, thr_of[name])}
        (r["pr_lo"], r["pr_hi"]), (r["roc_lo"], r["roc_hi"]) = cluster_boot(y_te, s, groups, args.n_boot, rng)
        rows.append(r)
        for fam in sorted(te["source_family"].unique()):
            k = (te["source_family"] == fam).to_numpy()
            if len(np.unique(y_te[k])) == 2:
                fam_rows.append({"model": name, "family": fam, **metrics(y_te[k], s[k], thr_of[name])})
    res = pd.DataFrame(rows)
    # collapse CNN seeds to mean (sd)
    res["base"] = res["model"].str.replace(r"__s\d+$", "", regex=True)
    agg = res.groupby("base").agg(
        n=("n", "first"), prevalence=("prevalence", "first"),
        pr_auc=("pr_auc", "mean"), pr_sd=("pr_auc", "std"), pr_lo=("pr_lo", "mean"), pr_hi=("pr_hi", "mean"),
        roc_auc=("roc_auc", "mean"), roc_sd=("roc_auc", "std"), roc_lo=("roc_lo", "mean"), roc_hi=("roc_hi", "mean"),
        f1=("f1", "mean"), runs=("model", "size")).reset_index().rename(columns={"base": "model"})
    cv = pd.read_csv(wd / "model_comparison_summary.csv") if (wd / "model_comparison_summary.csv").exists() else None
    if cv is not None:
        agg = agg.merge(cv[["model", "pr_auc_mean", "roc_auc_mean"]].rename(
            columns={"pr_auc_mean": "cv_pr_auc", "roc_auc_mean": "cv_roc_auc"}), on="model", how="left")
    agg["selected"] = agg["model"] == args.selected
    agg = agg.sort_values("pr_auc", ascending=False)
    agg.round(4).to_csv(wd / "holdout_results.csv", index=False)
    fam = pd.DataFrame(fam_rows)
    fam["model"] = fam["model"].str.replace(r"__s\d+$", "", regex=True)
    fam.groupby(["model", "family"]).mean(numeric_only=True).reset_index().round(4).to_csv(
        wd / "holdout_by_family.csv", index=False)
    pd.DataFrame({k: v for k, v in test_scores.items()}).assign(
        file_name=te["file name"], name=te["name"], group=te["group"],
        family=te["source_family"], label=y_te).to_csv(wd / "holdout_predictions.csv", index=False)

    print(f"\nHELD-OUT TEST (chance PR-AUC {y_te.mean():.3f}); 95% CI = signer-group bootstrap")
    print(agg[["model", "pr_auc", "pr_lo", "pr_hi", "roc_auc", "f1"]
              + (["cv_pr_auc"] if cv is not None else [])].round(3).to_string(index=False))

    if ext:
        erows = []
        for name, s in ext_scores.items():
            for subset, k in [("all", np.ones(len(s), bool)), ("signer-disjoint", ~ext["shared"])]:
                erows.append({"model": re.sub(r"__s\d+$", "", name), "subset": subset,
                              **metrics(ext["y"][k], s[k], thr_of[name])})
        edf = pd.DataFrame(erows).groupby(["model", "subset"]).mean(numeric_only=True).reset_index()
        edf.round(4).to_csv(wd / "external_results.csv", index=False)
        print("\nEXTERNAL 686-SET (human labels)")
        print(edf.round(3).to_string(index=False))

## Run

In [4]:
from types import SimpleNamespace as NS

run_holdout(NS(
    workdir=".",
    selected="SVM (RBF)",      # fixed before looking at test results
    skip_cnn=False,
    img_size=128, batch_size=32, epochs=25,
    cnn_variants="raw,norm",
    no_amp=False,
    n_boot=2000,
    external_xlsx=None,        # e.g. "Final_Assignments_relabeled.xlsx"
    external_dir=None,         # e.g. "signatures"
))

Using cached images from signature_images_256.npy (1289, 256, 256)
Fit on dev: 1035 rows / 89 groups; test: 254 rows / 23 groups, legible 0.291
  Majority baseline fit and scored
  Logistic (contrast only) fit and scored
  Logistic regression fit and scored
  Random forest fit and scored
  SVM (RBF) fit and scored
  Gradient boosting fit and scored
  CNN (raw)__s42 done
  CNN (raw)__s43 done
  CNN (raw)__s44 done
  CNN (norm)__s42 done
  CNN (norm)__s43 done
  CNN (norm)__s44 done

HELD-OUT TEST (chance PR-AUC 0.291); 95% CI = signer-group bootstrap
                   model  pr_auc  pr_lo  pr_hi  roc_auc    f1  cv_pr_auc
              CNN (norm)   0.574  0.369  0.788    0.801 0.617      0.401
               CNN (raw)   0.558  0.369  0.802    0.793 0.539      0.412
           Random forest   0.541  0.332  0.763    0.756 0.184      0.392
Logistic (contrast only)   0.493  0.234  0.732    0.692 0.225      0.453
       Gradient boosting   0.493  0.268  0.729    0.714 0.493      0.368
      